# 06 — Explainability (SHAP) & Walk-Forward Validation

- **Summary plot** — глобальная важность фичей (bar + beeswarm)
- **Waterfall plot** — объяснение одного предсказания (AAPL)
- **Group contributions** — вклад групп: tech / macro / sector / price / fundamental / sentiment
- **Dependence plots** — топ-3 признака
- **Walk-forward AUC** — стабильность модели во времени (5 фолдов)

In [1]:
import sys, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings('ignore')
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'backend'))

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'models'

from app.features.constants import (
    TECHNICAL_FEATURES, LAG_FEATURES, MACRO_FEATURES,
    SECTOR_FEATURES, PRICE_FEATURES, FUNDAMENTAL_FEATURES, SENTIMENT_FEATURES,
)

best_model = joblib.load(MODELS_DIR / 'model_xgb_optuna.joblib')
MODEL_FEATURES = best_model.get_booster().feature_names
print(f'Модель: XGBoost Optuna  |  Фичей: {len(MODEL_FEATURES)}')
print(f'shap: {shap.__version__}')

Модель: XGBoost Optuna  |  Фичей: 57
shap: 0.51.0


## 1. Загрузка данных

In [1]:
TICKERS = [
    'AAPL','MSFT','GOOGL','AMZN','NVDA','META','TSLA','ORCL','CRM','ADBE',
    'AMD','INTC','QCOM','TXN','JPM','BAC','GS','MS','WFC','BLK','AXP',
    'JNJ','UNH','PFE','ABT','MRK','TMO','WMT','HD','MCD','KO','PG','COST',
    'NKE','XOM','CVX','CAT','BA','HON','T','VZ','DIS','V','MA',
]

frames = []
for t in TICKERS:
    path = PROCESSED_DIR / t / 'technical.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        df['date'] = pd.to_datetime(df['date'])
        df['ticker'] = t
        frames.append(df.sort_values('date').reset_index(drop=True))

combined = pd.concat(frames, ignore_index=True).sort_values('date').reset_index(drop=True)
combined['fwd_return'] = combined.groupby('ticker')['returns'].transform(
    lambda x: x.rolling(5).sum().shift(-5))
combined = combined.dropna(subset=['fwd_return','returns'])
combined['target'] = combined.groupby('date')['fwd_return'].rank(pct=True).gt(0.75).astype(int)

for feat in ['returns','returns_3d','returns_10d','returns_20d','rsi','momentum',
             'volatility','volume_ratio','macd_hist','bb_pct','dist_52w_high','obv_change']:
    if feat in combined.columns:
        combined[f'{feat}_rank'] = combined.groupby('date')[feat].rank(pct=True)

avail = [f for f in MODEL_FEATURES if f in combined.columns]
split_idx = int(len(combined) * 0.8)
test  = combined.iloc[split_idx:]
X_test = test[avail].fillna(0)
y_test = test['target']
print(f'Фичей: {len(avail)}, Test: {len(X_test)} строк  pos={y_test.mean():.3f}')

Фичей: 57, Test: 10911 строк  pos=0.250


## 2. SHAP — глобальная важность фичей (Summary Bar)

In [1]:
np.random.seed(42)
idx = np.random.choice(len(X_test), size=300, replace=False)
X_sample = X_test.iloc[idx]

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)
avg_abs     = np.abs(shap_values).mean(axis=0)
feat_idx_map = {f: i for i, f in enumerate(avail)}
print(f'SHAP values: {shap_values.shape}')

fig, _ = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=avail, plot_type='bar', show=False, color='#6366f1')
plt.title('SHAP Feature Importance — XGBoost Optuna (Top-25% cross-sectional target)')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'shap_summary_bar.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()

SHAP values: (300, 57)


## 3. SHAP Beeswarm — влияние значений признаков

In [1]:
fig, _ = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=avail, show=False, max_display=20)
plt.title('SHAP Beeswarm — красный = высокое значение фичи, синий = низкое')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'shap_beeswarm.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()

## 4. Waterfall — объяснение предсказания AAPL

In [1]:
aapl_row = test[test['ticker'] == 'AAPL'].tail(1)
X_single = aapl_row[avail].fillna(0)
sv_single = explainer.shap_values(X_single)[0]
pred_prob = float(best_model.predict_proba(X_single)[0, 1])

shap_exp = shap.Explanation(
    values=sv_single,
    base_values=float(explainer.expected_value),
    data=X_single.iloc[0].values,
    feature_names=avail,
)
plt.figure(figsize=(12, 7))
shap.plots.waterfall(shap_exp, max_display=15, show=False)
plt.title(f'Waterfall: AAPL {aapl_row["date"].iloc[0].date()} | P(top-25%) = {pred_prob:.3f}')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'shap_waterfall.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()
print(f'Предсказание: P(top-25%) = {pred_prob:.3f}')

Предсказание: P(top-25%) = 0.488


## 5. Group Contributions — вклад групп фичей

In [1]:
CROSS_FEATURES = [f for f in avail if f.endswith('_rank')]
FEATURE_GROUPS = {
    'Technical':         set(TECHNICAL_FEATURES) | set(LAG_FEATURES),
    'Macro':             set(MACRO_FEATURES),
    'Sector':            set(SECTOR_FEATURES),
    'Price patterns':    set(PRICE_FEATURES),
    'Fundamental':       set(FUNDAMENTAL_FEATURES),
    'Sentiment':         set(SENTIMENT_FEATURES),
    'Cross-sect. ranks': set(CROSS_FEATURES),
}
total = avg_abs.sum()
group_scores = {}
for group, feat_set in FEATURE_GROUPS.items():
    idxs = [feat_idx_map[f] for f in feat_set if f in feat_idx_map]
    group_scores[group] = float(avg_abs[idxs].sum() / total) if total > 0 else 0.0

print('Group Contributions (avg |SHAP| share):')
for g, v in sorted(group_scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 35)
    print(f'  {g:<25}: {v:.1%}  {bar}')

colors = ['#6366f1','#22c55e','#f59e0b','#06b6d4','#a855f7','#ef4444','#64748b']
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(list(group_scores.keys()), list(group_scores.values()), color=colors, width=0.6)
ax.set_ylabel('Доля avg |SHAP|')
ax.set_title('Вклад групп признаков (XGBoost Optuna)')
ax.tick_params(axis='x', rotation=15); ax.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, group_scores.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{val:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'shap_group_contributions.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()

Group Contributions (avg |SHAP| share):
  Technical               : 28.1%  ██████████
  Cross-sect. ranks       : 24.1%  ████████
  Price patterns          : 16.8%  █████
  Fundamental             : 16.5%  █████
  Sentiment               : 8.1%   ██
  Sector                  : 4.8%   █
  Macro                   : 1.6%   


## 6. Dependence Plots — топ-3 признака

In [1]:
top3 = [avail[i] for i in np.argsort(avg_abs)[::-1][:3]]
print(f'Топ-3 признака: {top3}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, fname in zip(axes, top3):
    fi = feat_idx_map[fname]
    vals = shap_values[:, fi]
    sc = ax.scatter(X_sample[fname], vals, c=vals, cmap='RdYlGn', alpha=0.5, s=15,
                    vmin=np.percentile(vals, 5), vmax=np.percentile(vals, 95))
    ax.axhline(0, color='white', alpha=0.3)
    ax.set_xlabel(fname, fontsize=9); ax.set_ylabel('SHAP value', fontsize=9)
    ax.set_title(fname, fontsize=10)
    plt.colorbar(sc, ax=ax)
plt.suptitle('Dependence Plots — топ-3 признака', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'shap_dependence.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()

Топ-3 признака: ['volatility_rank', 'dist_52w_low', 'revenue_growth']


## 7. Walk-Forward Validation — стабильность AUC во времени

In [1]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

X_full = combined[avail].fillna(0)
y_full = combined['target']
tscv   = TimeSeriesSplit(n_splits=5, gap=5)

wf_results = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[tr_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[tr_idx], y_full.iloc[val_idx]
    pos = int(y_tr.sum()); neg = int((y_tr == 0).sum())
    clf = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.08,
                        scale_pos_weight=neg/pos if pos > 0 else 1,
                        eval_metric='logloss', random_state=42)
    clf.fit(X_tr, y_tr)
    auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    dates = combined['date'].iloc[val_idx]
    wf_results.append({'fold': fold+1, 'val_start': str(dates.min().date()),
                       'val_end': str(dates.max().date()), 'auc': round(auc, 4)})
    print(f'Fold {fold+1}: {dates.min().date()} – {dates.max().date()}  AUC={auc:.4f}')

wf_df = pd.DataFrame(wf_results)
print(f'\nMean AUC = {wf_df["auc"].mean():.4f}  (±{wf_df["auc"].std():.4f})')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wf_df['fold'], wf_df['auc'], 'o-', color='#6366f1', lw=2, markersize=8, label='Val AUC')
ax.axhline(wf_df['auc'].mean(), color='#22c55e', linestyle='--', lw=1.5,
           label=f'Mean = {wf_df["auc"].mean():.4f}')
ax.axhline(0.5, color='#94a3b8', linestyle=':', lw=1, label='Random (0.5)')
ax.fill_between(wf_df['fold'],
                wf_df['auc'].mean()-wf_df['auc'].std(),
                wf_df['auc'].mean()+wf_df['auc'].std(),
                alpha=0.15, color='#6366f1', label='±1 std')
ax.set_xlabel('Fold'); ax.set_ylabel('ROC-AUC')
ax.set_title('Walk-Forward Validation — XGBoost (5 folds, expanding window, gap=5d)')
ax.legend(fontsize=9); ax.grid(alpha=0.2); ax.set_ylim(0.45, 0.70)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'walk_forward_auc.png', dpi=100, bbox_inches='tight')
plt.show(); plt.close()
print(wf_df.to_string(index=False))

Fold 1: 2022-02-08 – 2022-12-05  AUC=0.5123
Fold 2: 2022-12-05 – 2023-10-02  AUC=0.5134
Fold 3: 2023-10-02 – 2024-07-30  AUC=0.5333
Fold 4: 2024-07-30 – 2025-05-28  AUC=0.5587
Fold 5: 2025-05-28 – 2026-03-24  AUC=0.5555

Mean AUC = 0.5346  (±0.0222)

 fold   val_start     val_end     auc
     1  2022-02-08  2022-12-05  0.5123
     2  2022-12-05  2023-10-02  0.5134
     3  2023-10-02  2024-07-30  0.5333
     4  2024-07-30  2025-05-28  0.5587
     5  2025-05-28  2026-03-24  0.5555
